In [2]:
import pandas as pd

In [ ]:
# load the train dataset
df = pd.read_csv("../data/train/balanced_train.csv")

In [19]:
import pandas as pd
import json
import ast

def prepare_training_data(csv_path, output_path):
    df = pd.read_csv(csv_path)
    
    # 1. Clean nulls
    df = df[df['is_supported'].notna()]
    df['confidence'] = df.apply(
        lambda row: 1 if pd.isna(row['confidence']) else int(row['confidence']),
        axis=1
    )
    
    # 2. Balance classes (undersample majority)
    supported = df[df['is_supported'] == True]
    unsupported = df[df['is_supported'] == False]
    
    # Keep all supported, sample equal unsupported
    n_supported = len(supported)
    unsupported_sampled = unsupported.sample(n=n_supported, random_state=42)
    
    df_balanced = pd.concat([supported, unsupported_sampled]).sample(frac=1, random_state=42)
    
    print(f"Balanced dataset: {len(df_balanced)} samples")
    print(f"Supported: {len(df_balanced[df_balanced['is_supported'] == True])}")
    print(f"Unsupported: {len(df_balanced[df_balanced['is_supported'] == False])}")
    
    # 3. Convert to instruction format
    training_data = []
    
    for idx, row in df_balanced.iterrows():

        print(row)
        # Parse string representations if needed
        try:
            text_triplets = ast.literal_eval(row['document_triplets']) if isinstance(row['document_triplets'], str) else row['document_triplets']
            summary_triple = ast.literal_eval(row['summary_triplet_full']) if isinstance(row['summary_triplet_full'], str) else row['summary_triplet_full']
            supporting_evidence = ast.literal_eval(row['supporting_evidence']) if isinstance(row['supporting_evidence'], str) else row['supporting_evidence']
        except:
            continue
        
        # Format user input
        user_content = f"""Text triplets: {json.dumps(text_triplets)}

                            Summary triple: {json.dumps(summary_triple)}

                            Verify if this summary triple is supported by the text triplets."""
        
        # Instruction (same for all examples)
        instruction = "Given a document with text triplets and a summary triple, determine if the summary triple is supported by the document's text triplets. Construct the knowledge graph and traverse it to find supporting evidence. Provide a detailed response in JSON format with fields: summary_triple, is_supported, supporting_evidence, confidence, and reasoning."
        
        # Input (the actual task data)
        input_text = f"Text triplets: {json.dumps(text_triplets)}\n\nSummary triple: {json.dumps(summary_triple)}"
        
        # Output (the expected response)
        output = {
            "summary_triple": summary_triple,
            "is_supported": bool(row['is_supported']),
            "supporting_evidence": supporting_evidence if supporting_evidence else [],
            "confidence": int(row['confidence']) if row['is_supported'] else None,
            "reasoning": str(row['reasoning'])
        }
        output_text = json.dumps(output, ensure_ascii=False)
        
        training_data.append({
            "instruction": instruction,
            "input": input_text,
            "output": output_text
        })

        
        # # Format assistant response
        # response = {
        #     "summary_triple": summary_triple,
        #     "is_supported": bool(row['is_supported']),
        #     "supporting_evidence": supporting_evidence if supporting_evidence else [],
        #     "confidence": int(row['confidence']) if row['is_supported'] else None,
        #     "reasoning": str(row['reasoning'])
        # }
        
        # conversation = {
        #     "conversations": [
        #         {
        #             "role": "system",
        #             "content": "You are a helpful assistant in traversing and reasoning about knowledge graphs. Given a document with text triplets and a summary triple, determine if the summary triple is supported by the document's text triplets. Construct the knowledge graph and traverse it to find supporting evidence. Provide a detailed response in JSON format."
        #         },
        #         {
        #             "role": "user",
        #             "content": user_content
        #         },
        #         {
        #             "role": "assistant",
        #             "content": json.dumps(response, ensure_ascii=False)
        #         }
        #     ]
        # }
        
        # training_data.append(conversation)

    # print(training_data)
    
    # 4. Split train/validation (85/15)
    split_idx = int(len(training_data) * 0.85)
    train_data = training_data[:split_idx]
    val_data = training_data[split_idx:]
    
    # 5. Save as JSONL
    with open(f"{output_path}/train.jsonl", 'w', encoding='utf-8') as f:
        for item in train_data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    
    with open(f"{output_path}/validation.jsonl", 'w', encoding='utf-8') as f:
        for item in val_data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    
    print(f"\nSaved {len(train_data)} training samples")
    print(f"Saved {len(val_data)} validation samples")
    
    return train_data, val_data

# Usage
train, val = prepare_training_data('../data/train/expanded_triplet_evaluations.csv', '../data/train')

Balanced dataset: 700 samples
Supported: 350
Unsupported: 350
doc_id                                                                     120
document                     This is according to a study revealing that 96...
summary                      The Madagascar pochard, the world's rarest bir...
document_triplets            [['chicks', 'dying at', 'two to three weeks ol...
gen_summary_triplets         [['Madagascar pochard', 'is', "world's rarest ...
summary_triplet_subject                                     Madagascar pochard
summary_triplet_predicate                                                needs
summary_triplet_object                                        new wetland home
summary_triplet_full         ['Madagascar pochard', 'needs', 'new wetland h...
is_supported                                                              True
confidence                                                                   2
reasoning                    The Text Triplet explicitly states that 